In [1]:
!pip -q install tensorflow==2.* gradio pillow

# Imports
import os
import io
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from google.colab import files
from PIL import Image
import gradio as gr

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [2]:
print("Version 9 from Jenkins 🚀")

Version 9 from Jenkins 🚀


In [2]:
# Importing JSON and .H5 files

import os
from google.colab import files

!pip install -q google-cloud-vision

#  Step 1: Upload Google Cloud Vision credentials
print("Please upload your Google Cloud Vision credentials JSON.")
print("   → If you skip this, OCR will be disabled and only the CNN model will run.")

json_upload = files.upload()

CREDENTIALS_PATH = None
for fname in json_upload.keys():
    if fname.lower().endswith(".json"):
        CREDENTIALS_PATH = fname
        break

if CREDENTIALS_PATH:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS_PATH
    print(f"GOOGLE_APPLICATION_CREDENTIALS set to: {CREDENTIALS_PATH}")
else:
    print("ℹ️ No valid .json credentials uploaded — continuing without GCP credentials (OCR will be skipped).")

# Step 2: Upload CNN model (.h5)
MODEL_PATH = None
while MODEL_PATH is None:
    print(" Please upload your trained CNN model file (.h5).")
    model_upload = files.upload()
    for fname in model_upload.keys():
        if fname.lower().endswith(".h5"):
            MODEL_PATH = fname
            break
    if MODEL_PATH is None:
        print("No .h5 file detected. Please try again.")

print(f"Model file detected: {MODEL_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/543.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.2/543.2 kB 18.4 MB/s eta 0:00:00
Please upload your Google Cloud Vision credentials JSON.
   → If you skip this, OCR will be disabled and only the CNN model will run.


Saving cedar-fragment-471717-s0-6878f1e845a0.json to cedar-fragment-471717-s0-6878f1e845a0.json
GOOGLE_APPLICATION_CREDENTIALS set to: cedar-fragment-471717-s0-6878f1e845a0.json
 Please upload your trained CNN model file (.h5).


Saving brand_detection_model.h5 to brand_detection_model.h5
Model file detected: brand_detection_model.h5


In [3]:
# Core Logic (OCR → fallback to CNN)

import re
from difflib import SequenceMatcher
from google.cloud import vision

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

try:
    _, H, W, C = model.input_shape
    IMG_SIZE = (H, W)
except Exception:
    IMG_SIZE = (224, 224)

print("Using input size:", IMG_SIZE)

CLASS_NAMES = ['Adidas', 'Apple', 'Nike', 'Swarovski', 'Under Armour']

_dummy = np.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)
_out = model.predict(_dummy, verbose=0)
_num_classes = int(_out.shape[-1])
if _num_classes != len(CLASS_NAMES):
    raise ValueError(f"Model outputs {_num_classes} classes, but CLASS_NAMES has {len(CLASS_NAMES)}.")

BRANDS = [
    "Adidas","Adidas Kids","Adidas Originals","Apple","Armani","Armani Exchange","Asics","Boss",
    "Brooks Brothers","Byredo","Calvin Klein","Calvin Klein Underwear","Chanel","Charles Tyrwhitt",
    "Coach","Crocs","Da Milano","Diesel","Dior","Dune London","Ed-A-Mamma","Emporio Armani",
    "Ethos Summit","Forest Essentials","Forever New","Freshpik@Jio World Drive","Gant","GAS",
    "Hamleys","Hamleys Play","Heads Up For Tails","Hunkemöller","Jean-Claude Biguine","Jo Malone",
    "Kama Ayurveda","Kate Spade","LensCrafters","MAC","Maison Des Parfums","Maje",
    "Marks & Spencer Lingerie","Michael Kors","Montblanc","Mothercare","Mulmul","Needledust",
    "Nike","Onitsuka Tiger","Paul Smith","Ritu Kumar","Samsonite","Sandro","Satya Paul",
    "Scotch & Soda","Steve Madden","Sunglass Hut","Superdry","Superdry Sport","Swarovski",
    "The White Crow","The White Crow - Books & Coffee","Tira","Tommy Hilfiger","Tommy Hilfiger Kids",
    "Tumi","Under Armour","Vero Moda","West Elm","Anna Rasoii","Arbab","Asian Bistro By HOM","Bateel",
    "Burma Burma","Cavu By Nolita","CocoCart","Cou Cou By Oberoi","Entisi","FOO","Genda Phool",
    "Grandmama's Cafe","Harajuku Bakehouse","Harajuku Tokyo Café & Bar",
    "Hurrem's Turkish Baklava & Confectionery","Hylo Taproom By Igloo","Indigo Delicatessen","Kofuku",
    "MOTODO","Neel","Pret A Manger","SAZ","Someplace Else","Spice-O-Pedia","Starbucks",
    "The Nutcracker","The White Crow Books And Coffee","Twisting Scoops"
]

# Text normalization

def _canon(s: str) -> str:
    s = s.lower()
    s = re.sub(r"['’`]", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def best_brand_match_from_text(text: str, brands, threshold: float = 0.80):
    """Return brand if OCR text matches one from list."""
    if not text:
        return None
    ct = _canon(text)
    best_brand, best_score = None, 0.0
    for b in brands:
        cb = _canon(b)
        if cb in ct or ct in cb:
            score = 1.0 if cb in ct else 0.9
        else:
            score = SequenceMatcher(None, cb, ct).ratio()
        if score > best_score:
            best_brand, best_score = b, score
    if best_brand and best_score >= threshold:
        return best_brand
    return None

def ocr_detect_brand(img_pil: Image.Image):


    if not os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):

        return None
    try:
        client = vision.ImageAnnotatorClient()
        buf = io.BytesIO()
        img_pil.convert("RGB").save(buf, format="PNG")
        img = vision.Image(content=buf.getvalue())
        resp = client.text_detection(image=img)
        if resp.error.message:
            print("Vision API error:", resp.error.message)
            return None
        anns = resp.text_annotations
        if not anns:
            # print("OCR: no text detected.")
            return None
        full_text = anns[0].description
        # print("OCR text:", full_text)
        return best_brand_match_from_text(full_text, BRANDS)
    except Exception as e:
        print(" OCR skipped:", repr(e))
        return None

# CNN Prediction
def preprocess_pil(img_pil: Image.Image, target_size=IMG_SIZE):
    img = img_pil.convert("RGB").resize(target_size)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0).astype("float32") / 255.0
    return arr

def cnn_predict_pil(img_pil: Image.Image):
    arr = preprocess_pil(img_pil, IMG_SIZE)
    preds = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(preds))
    conf = float(np.max(preds))
    return CLASS_NAMES[idx], conf

# Unified Prediction Logic
def predict_brand(img_pil: Image.Image):
    """Check OCR first → if no match → CNN fallback."""
    ocr_brand = ocr_detect_brand(img_pil)
    if ocr_brand:
        print(f"Predicted Brand: {ocr_brand}")
        return ocr_brand, 1.0, "OCR"
    else:
        label, conf = cnn_predict_pil(img_pil)
        print(f"Predicted Brand: {label}")
        return label, conf, "CNN"

print("Logic ready (OCR → CNN fallback).")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 178, 178, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 89, 89, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 87, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 43, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 41, 41, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 18, 18, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 10368)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │     1,327,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,568,711 (5.98 MB)

 Trainable params: 1,568,709 (5.98 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Using input size: (180, 180)
Logic ready (OCR → CNN fallback).


In [4]:
# Upload a single test image and run one prediction (OCR → CNN fallback)
print("📷 Upload a test image (jpg/png).")
test_up = files.upload()

test_img_path = None
for fname in test_up.keys():
    if fname.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        test_img_path = fname
        break

if test_img_path is None:
    raise FileNotFoundError("No image file uploaded.")

# Open the uploaded image
img = Image.open(test_img_path)

# Run prediction using unified logic
label, conf, method = predict_brand(img)

# Display results
print("\n Predicted Brand:", label)

img.show()

📷 Upload a test image (jpg/png).


Saving image.jpg to image.jpg
 OCR skipped: PermissionDenied('This API method requires billing to be enabled. Please enable billing on project #628760278283 by visiting https://console.developers.google.com/billing/enable?project=628760278283 then retry. If you enabled billing for this project recently, wait a few minutes for the action to propagate to our systems and retry.')
Predicted Brand: Apple

 Predicted Brand: Apple


In [ ]:
# Gradio UI
import gradio as gr
from PIL import Image
import numpy as np

def gradio_infer(img):
    if img is None:
        return "No image"

    img_pil = Image.fromarray(img.astype("uint8"))
    result = predict_brand(img_pil)

    # Handle tuple or direct string outputs safely
    if isinstance(result, tuple) and len(result) > 0:
        label = result[0]
    else:
        label = str(result)

    return label

demo = gr.Interface(
    fn=gradio_infer,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=gr.Textbox(label="Predicted Brand"),
    title="Brand Detection (OCR → CNN)",
    description="OCR tries to match your brand list first (100% if match). If not, falls back to CNN prediction.",
    allow_flagging="never"
)

demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc21b256089fcc4e9e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
